# 03 - MCP 入門：製作你的第一個 MCP Server

**MCP（Model Context Protocol）** 是 Anthropic 制定的開放協定，讓 LLM 應用程式能透過標準化介面連接外部工具與資料。
任何支援 MCP 的 Client（如 Claude Desktop、Cursor）都能自動發現並呼叫你的 Server 所暴露的功能。

一個 MCP Server 可以暴露三種原語（primitives）給 LLM：

| 原語 | 裝飾器 | 用途 |
|------|--------|------|
| **Tools** | `@mcp.tool()` | LLM 主動呼叫的函式（有副作用） |
| **Resources** | `@mcp.resource(uri)` | LLM 可讀取的靜態/動態資料 |
| **Prompts** | `@mcp.prompt()` | 可重用的提示模板 |

In [13]:
import sys
from pathlib import Path

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

import utils  # noqa: F401 — suppresses httpx HTTP request logs

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mistin/research/agentic-rag-lab


## 架構概覽

```
MCP Client
(Claude Desktop / Cursor / 任何相容 App)
        ↕  MCP Protocol（stdio 或 SSE）
MCP Server
(你的 Python server.py)
        ↕
Tools / Resources / Local Data / APIs
```

- **stdio 模式**（預設）：Client 以子行程方式啟動 Server，透過 stdin/stdout 通訊，適合本地工具。
- **SSE 模式**：Server 作為 HTTP 服務運行，適合遠端部署或多 Client 場景。

## 1 · 建立 FastMCP 實例

`FastMCP` 是 MCP Python SDK 提供的高階包裝，讓你用裝飾器直接定義工具，無需手動處理協定細節。

In [14]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")
print(f"Server name: {mcp.name}")

Server name: demo-server


## 2 · 定義 Tools

`@mcp.tool()` 將普通 Python 函式包裝成 LLM 可呼叫的工具：
- **docstring** → 工具說明（LLM 據此決定何時呼叫）
- **型別標注** → input schema（SDK 自動透過 Pydantic 生成 JSON Schema）
- **回傳值** → 工具執行結果，傳回給 LLM 繼續推理

In [15]:
@mcp.tool()
def add(a: int, b: int) -> int:
    """將兩個整數相加並回傳結果。"""
    return a + b


@mcp.tool()
def greet(name: str) -> str:
    """以繁體中文問候指定的人。"""
    return f"你好，{name}！歡迎使用 MCP。"

### 直接測試 Tools

Tool 函式本質上就是普通 Python 函式，可以直接在 Notebook 中呼叫，不需要啟動 Server。

In [16]:
print(add(3, 4))
print(greet("世界"))

7
你好，世界！歡迎使用 MCP。


## 3 · 定義 Resources

`@mcp.resource(uri)` 暴露資料給 LLM 讀取。Resources 適合靜態說明文件、設定檔，或動態查詢的資料集。

URI 可以是固定字串（靜態），也可以是帶有參數的模板（動態），例如 `"db://users/{user_id}"`。

In [17]:
@mcp.resource("info://server-description")
def server_description() -> str:
    """這個 demo-server 的簡介與可用工具清單。"""
    return (
        "demo-server 是一個示範用的 MCP Server。\n"
        "可用工具：\n"
        "  - add(a, b)   : 整數加法\n"
        "  - greet(name) : 中文問候\n"
        "可用資源：\n"
        "  - info://server-description : 本說明文字"
    )


# Resource 函式同樣可以直接呼叫
print(server_description())

demo-server 是一個示範用的 MCP Server。
可用工具：
  - add(a, b)   : 整數加法
  - greet(name) : 中文問候
可用資源：
  - info://server-description : 本說明文字


## 4 · 查看已註冊工具的 Schema

MCP SDK 根據函式的型別標注自動生成 JSON Schema，這份 schema 會在 Server 啟動時傳給 Client，讓 LLM 知道每個工具的參數格式。

In [18]:
import inspect

for fn in [add, greet]:
    sig = inspect.signature(fn)
    params = {
        name: str(param.annotation.__name__)
        for name, param in sig.parameters.items()
    }
    print(f"@mcp.tool: {fn.__name__}")
    print(f"  docstring  : {fn.__doc__}")
    print(f"  parameters : {params}")
    print(f"  return type: {sig.return_annotation.__name__}")
    print()

@mcp.tool: add
  docstring  : 將兩個整數相加並回傳結果。
  parameters : {'a': 'int', 'b': 'int'}
  return type: int

@mcp.tool: greet
  docstring  : 以繁體中文問候指定的人。
  parameters : {'name': 'str'}
  return type: str



## 5 · 獨立 Server 檔案（examples/mcp/mcp_server.py）

03 與 04 共用同一個 server，放在 `examples/mcp/mcp_server.py`。
Server 以 **SSE 模式**啟動（HTTP，預設 port 3100），不再透過 stdio 子行程，
Client 透過 HTTP 連線，因此 server 必須事先手動啟動。

In [19]:
MCP_SERVER = EXAMPLES_DIR / "mcp" / "mcp_server.py"
print(f"MCP server   : {MCP_SERVER}")
print()
src = MCP_SERVER.read_text()
# 只顯示前 40 行供概覽
for line in src.splitlines()[:40]:
    print(line)


MCP server   : /home/mistin/research/agentic-rag-lab/examples/mcp/mcp_server.py

#!/usr/bin/env python
"""統一 MCP Server — 供 03 與 04 notebook 共用。

Tools:
  add(a, b)                            整數加法（03 示範用）
  greet(name)                          中文問候（03 示範用）
  search_meetings(pattern, max_results)  搜尋 CRM 會議紀錄（04 示範用）
  read_meeting(filename)               讀取完整會議紀錄（04 示範用）

Resources:
  info://server-description            本 server 的工具清單說明
  crm://customers                      CRM 客戶名稱清單（動態萃取自檔名）
  crm://tags                           CRM 標籤 taxonomy

Prompts:
  analyze_customer(customer_name)      分析指定客戶的 prompt 模板
"""
import sys
from pathlib import Path

# utils/ 在 examples/ 下，從 examples/mcp/ 往上一層找
sys.path.insert(0, str(Path(__file__).parent.parent))

from mcp.server.fastmcp import FastMCP

from utils.tools import FileTools

MCP_HOST = "127.0.0.1"
MCP_PORT = 3100

mcp = FastMCP("crm-demo-server", host=MCP_HOST, port=MCP_PORT)
_crm = FileTools(Path(__file__).parent.parent.parent / "da

### 啟動 MCP Server

在**另一個終端機**執行下列指令，server 會在 `http://127.0.0.1:3100/sse` 監聽 SSE 連線：

```bash
cd /path/to/agentic-rag-lab
uv run examples/mcp/mcp_server.py
# 輸出：MCP server 啟動於 http://127.0.0.1:3100/sse
```

server 啟動後保持運行，再執行下面的 cells 即可連線。

> ⚠️ **`mcp dev` 只適合瀏覽器視覺化測試，不能作為 SSE client 的連接目標。**
> `uv run mcp dev examples/mcp/mcp_server.py` 會啟動 Inspector UI（預設 port 6274），
> 但 `/sse` 路徑在 Inspector 上回傳的是 HTML，不是 SSE stream，會出現
> `Expected Content-Type 'text/event-stream', got 'text/html'` 錯誤。
> 要讓 notebook cells 正常運作，請使用 `python examples/mcp/mcp_server.py`（port 3100）。

## 6 · 使用 LLM 呼叫 MCP Tools

以下示範完整的 **MCP Client ↔ LLM 整合流程**：

```
Notebook（Client）
  1. 連線到已啟動的 MCP server（SSE，http://127.0.0.1:3100/sse）
  2. 取得 server 的 tool list → 轉成 OpenAI function schema
  3. 把 schema 傳給 LLM → LLM 決定呼叫哪個工具
  4. 透過 MCP 協定執行工具 → 取得結果
  5. 把結果送回 LLM → 取得最終回覆
```

> LLM 設定來自 `.env`（`VLLM_BASE_URL` / `VLLM_API_KEY` / `VLLM_MODEL`）。

In [20]:
import os
import json
import re
from openai import OpenAI
from mcp import ClientSession
from mcp.client.sse import sse_client
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

llm = OpenAI(
    base_url=os.environ["VLLM_BASE_URL"],
    api_key=os.environ["VLLM_API_KEY"],
)
MODEL = os.environ["VLLM_MODEL"]
print(f"LLM: {MODEL}")

LLM: gemma-4-31B-it


### 連線到 MCP Server，列出可用工具

In [21]:
# Server 已在另一個終端機手動啟動（python examples/mcp/mcp_server.py）
SERVER_URL = "http://127.0.0.1:3100/sse"


async def list_mcp_tools():
    async with sse_client(SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools


mcp_tools = await list_mcp_tools()
for t in mcp_tools:
    print(f"  [{t.name}]    {t.description}")

  [add]    將兩個整數相加並回傳結果。
  [greet]    以繁體中文問候指定的人。
  [search_meetings]    在 CRM 會議紀錄中搜尋關鍵詞，回傳 'file:lineno: text' 清單。
  [read_meeting]    讀取指定的會議紀錄檔案，回傳完整內容。


### LLM + MCP 整合主函式

`run_with_mcp(question)` 實作完整的單輪工具呼叫循環。
同時處理兩種工具呼叫格式：
- **標準 `tool_calls`**：OpenAI function calling 格式
- **Gemma-4 fallback**：模型有時以純文字輸出 `call:func{key:val}` 格式的工具呼叫

In [22]:
_TEXT_TOOL_RE = re.compile(r"call:(\w+)\{([^}]*)\}")


def _try_cast(v: str):
    try:
        return int(v)
    except ValueError:
        pass
    try:
        return float(v)
    except ValueError:
        pass
    return v


def _parse_all_text_calls(content: str) -> list[tuple[str, dict]]:
    calls = []
    for m in _TEXT_TOOL_RE.finditer(content):
        args = {}
        for kv in m.group(2).split(","):
            if ":" in kv:
                k, v = kv.split(":", 1)
                args[k.strip()] = _try_cast(v.strip())
        calls.append((m.group(1), args))
    return calls


async def run_with_mcp(question: str) -> None:
    async with sse_client(SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Step 1: 取得 MCP tools → 轉成 OpenAI function schema
            result = await session.list_tools()
            openai_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in result.tools
            ]

            messages = [{"role": "user", "content": question}]
            print(f"Q: {question}\n")

            # Step 2: LLM 決定呼叫哪個工具
            resp = llm.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto",
            )
            msg = resp.choices[0].message

            # Gemma-4 fallback：模型以純文字格式輸出工具呼叫
            if not msg.tool_calls and isinstance(msg.content, str):
                text_calls = _parse_all_text_calls(msg.content)
                if text_calls:
                    tool_results = []
                    for name, args in text_calls:
                        print(f"→ MCP call : {name}({args})")
                        r = await session.call_tool(name, args)
                        out = r.content[0].text if r.content else ""
                        print(f"  MCP result: {out}")
                        tool_results.append(f"{name}({args}) → {out}")
                    messages.append({"role": "assistant", "content": msg.content})
                    messages.append({
                        "role": "user",
                        "content": "工具執行結果：\n" + "\n".join(tool_results) + "\n\n請根據以上結果回答原始問題。",
                    })
                    final = llm.chat.completions.create(model=MODEL, messages=messages)
                    print(f"\nA: {final.choices[0].message.content}")
                    return
                print(f"A: {msg.content}")
                return

            # 標準 tool_calls 路徑
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    }
                    for tc in msg.tool_calls
                ],
            })

            # Step 3: 透過 MCP 協定執行工具
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"→ MCP call : {tc.function.name}({args})")
                r = await session.call_tool(tc.function.name, args)
                out = r.content[0].text if r.content else ""
                print(f"  MCP result: {out}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

            # Step 4: LLM 整合工具結果，給出最終回覆
            final = llm.chat.completions.create(model=MODEL, messages=messages)
            print(f"\nA: {final.choices[0].message.content}")

### 執行示範

In [23]:
# 同時呼叫兩個工具
await run_with_mcp("請幫我計算 123 加 456，並用中文問候『世界』。")

Q: 請幫我計算 123 加 456，並用中文問候『世界』。

→ MCP call : add({'a': 123, 'b': 456})
  MCP result: 579
→ MCP call : greet({'name': '世界'})
  MCP result: 你好，世界！歡迎使用 MCP。

A: 123 加 456 的結果是 579。

你好，世界！


In [24]:
# 只需要加法
await run_with_mcp("999 加 1 等於多少？")

Q: 999 加 1 等於多少？

→ MCP call : add({'a': 999, 'b': 1})
  MCP result: 1000

A: 999 加 1 等於 1000。


## 小結

本筆記本涵蓋了 MCP Server 的完整開發與整合流程：

1. `uv pip install mcp` — 安裝 SDK
2. `FastMCP(name, host, port)` — 建立 SSE server 實例
3. `@mcp.tool()` — 定義 LLM 可呼叫的函式（docstring + type hints 自動生成 schema）
4. `@mcp.resource(uri)` — 暴露資料給 LLM 讀取
5. `mcp.run(transport="sse")` — 以 SSE 模式啟動 server（`examples/mcp/mcp_server.py`）
6. `ClientSession` + `sse_client` — 從 Python 以 MCP 協定連線至 server
7. LLM 呼叫流程：取得 tool schema → 送給 LLM → 執行 MCP tool → 回傳結果 → 最終回覆

---

**下一步可以探索：**
- `uv run mcp dev examples/mcp/mcp_server.py` — 開啟 MCP Inspector 進行互動測試
- `@mcp.prompt()` — 定義可重用的 prompt 模板（見 notebook 04）
- 動態 resource URI 模板：`@mcp.resource("db://users/{user_id}")`
- 多輪對話 + 多次工具呼叫（LangGraph ReAct loop）